In [108]:
import pandas as pd
import pylangacq as pla
import numpy as np 

# aphasiabank

In [109]:
file_path = '/Users/monicaromero/PycharmProjects/afasia_cat/aphasiabank_es/df_ES_clean_2.csv'
df_aphbank = pd.read_csv(file_path)
df_aphbank = df_aphbank[df_aphbank['file'] != "TCU06a.wav"] #este no tiene info del tipo de afasia
df_aphbank.head()

,mark_start,mark_end,transcriptions,sex,age,file,WAB_AQ,aphasia_type,WAB_AQ_category,fluency_speech,file_cut,duration,num_words
0,20585,24500,okay dímelo otra vez,male,42.0,TCU02a.wav,77.3,Anomic,Mild,Fluent,TCU02a_20.585_3.915.wav,3.915,4
1,25710,26400,sí,male,42.0,TCU02a.wav,77.3,Anomic,Mild,Fluent,TCU02a_25.71_0.69.wav,0.690,1
2,29400,31200,que piensa mi habla,male,42.0,TCU02a.wav,77.3,Anomic,Mild,Fluent,TCU02a_29.4_1.8.wav,1.800,4
3,33248,40362,no es que flr flr decir las cosas a vveces tengo problemas como ahorita,male,42.0,TCU02a.wav,77.3,Anomic,Mild,Fluent,TCU02a_33.248_7.114.wav,7.114,14
4,40362,47984,ahorita puedo hablar pero flr dos horas flr flr no puedo decir lo todo,male,42.0,TCU02a.wav,77.3,Anomic,Mild,Fluent,TCU02a_40.362_7.622.wav,7.622,14


In [110]:
def extract_patient_info(file_directory):
    ds = pla.read_chat(file_directory)
    files = ds.file_paths()

    # Listas para almacenar la información extraída
    v_sex = []
    v_age = []
    v_WAB_AQ = []
    v_aphasia_type = []
    v_WAB_AQ_category = []
    v_fluency_speech = []
    v_file_name = []

    for f in files:
        ds2 = pla.read_chat(f)
        header = ds2.headers()

        # Extraer información de los participantes
        sex = header[0]['Participants']['PAR']['sex']
        age = header[0]['Participants']['PAR']['age']
        WAB_AQ = header[0]['Participants']['PAR'].get('custom', None)
        aphasia_type = header[0]['Participants']['PAR']['group']

        # Extraer nombre del archivo
        file_name = f.split('/')[-1].replace('.cha', '.wav')

        # Añadir la información a las listas
        v_sex.append(sex)
        v_age.append(age[:2] if age else None)  # Quitar los meses
        v_WAB_AQ.append(WAB_AQ)
        v_aphasia_type.append(aphasia_type)
        v_file_name.append(file_name)

    # Crear un DataFrame con la información extraída
    df_info = pd.DataFrame({
        'file': v_file_name,
        'sex': v_sex,
        'age': v_age,
        'WAB_AQ': v_WAB_AQ,
        'aphasia_type': v_aphasia_type
    })

    # Aplicar reglas para WAB_AQ_category
    df_info.loc[(pd.to_numeric(df_info['WAB_AQ'], errors='coerce') >= 0) & (pd.to_numeric(df_info['WAB_AQ'], errors='coerce') <= 25), 'WAB_AQ_category'] = 'Very severe'
    df_info.loc[(pd.to_numeric(df_info['WAB_AQ'], errors='coerce') > 25) & (pd.to_numeric(df_info['WAB_AQ'], errors='coerce') <= 50), 'WAB_AQ_category'] = 'Severe'
    df_info.loc[(pd.to_numeric(df_info['WAB_AQ'], errors='coerce') > 50) & (pd.to_numeric(df_info['WAB_AQ'], errors='coerce') <= 75), 'WAB_AQ_category'] = 'Moderate'
    df_info.loc[(pd.to_numeric(df_info['WAB_AQ'], errors='coerce') > 75), 'WAB_AQ_category'] = 'Mild'
    df_info.loc[pd.to_numeric(df_info['WAB_AQ'], errors='coerce').isna(), 'WAB_AQ_category'] = 'Unknown'

    # Aplicar reglas para fluency_speech
    df_info.loc[df_info['aphasia_type'].isin(['Anomic', 'Conduction', 'Fluent', 'Wernicke', 'TransSensory']), 'fluency_speech'] = 'Fluent'
    df_info.loc[df_info['aphasia_type'].isin(['Broca', 'Global', 'TransMotor']), 'fluency_speech'] = 'Non Fluent'
    df_info.loc[df_info['aphasia_type'] == 'NotAphasicByWAB', 'fluency_speech'] = 'Unknown'

    return df_info

In [111]:
file_directory = '/Users/monicaromero/PycharmProjects/afasia_cat/aphasiabank_es'
df_info = extract_patient_info(file_directory)
df_info

,file,sex,age,WAB_AQ,aphasia_type,WAB_AQ_category,fluency_speech
0,TCU02a.wav,male,42,77.3,Anomic,Mild,Fluent
1,TCU04a.wav,male,48,83.8,Anomic,Mild,Fluent
2,TCU06a.wav,male,68,62.6,,Moderate,NaN
3,TCU10a.wav,female,53,77.6,,Mild,NaN


In [112]:
df_aphbank = pd.read_csv('/Users/monicaromero/PycharmProjects/afasia_cat/aphasiabank_es/df_ES_clean_2.csv')

# Eliminar el archivo "TCU06a.wav" si no tiene información
if 'file' in df_aphbank.columns:
    df_aphbank = df_aphbank[df_aphbank['file'] != "TCU06a.wav"]

# Combinar ambos datasets por la columna 'file'
df_combined = pd.merge(df_aphbank, df_info, on='file', how='left', suffixes=('', '_new'))

# Llenar los valores NaN del dataset original con los nuevos valores extraídos
df_combined['sex'] = df_combined['sex'].fillna(df_combined['sex_new'])
df_combined['age'] = df_combined['age'].fillna(df_combined['age_new'])
df_combined['WAB_AQ'] = df_combined['WAB_AQ'].fillna(df_combined['WAB_AQ_new'])
df_combined['aphasia_type'] = df_combined['aphasia_type'].fillna(df_combined['aphasia_type_new'])
df_combined['WAB_AQ_category'] = df_combined['WAB_AQ_category'].fillna(df_combined['WAB_AQ_category_new'])
df_combined['fluency_speech'] = df_combined['fluency_speech'].fillna(df_combined['fluency_speech_new'])

# Eliminar columnas adicionales generadas durante la combinación
df_combined = df_combined.drop(columns=[
    'sex_new', 'age_new', 'WAB_AQ_new', 'aphasia_type_new', 'WAB_AQ_category_new', 'fluency_speech_new'
])
df_combined

,mark_start,mark_end,transcriptions,sex,age,file,WAB_AQ,aphasia_type,WAB_AQ_category,fluency_speech,file_cut,duration,num_words
0,20585,24500,okay dímelo otra vez,male,42.0,TCU02a.wav,77.3,Anomic,Mild,Fluent,TCU02a_20.585_3.915.wav,3.915,4
1,25710,26400,sí,male,42.0,TCU02a.wav,77.3,Anomic,Mild,Fluent,TCU02a_25.71_0.69.wav,0.690,1
2,29400,31200,que piensa mi habla,male,42.0,TCU02a.wav,77.3,Anomic,Mild,Fluent,TCU02a_29.4_1.8.wav,1.800,4
3,33248,40362,no es que flr flr decir las cosas a vveces tengo problemas como ahorita,male,42.0,TCU02a.wav,77.3,Anomic,Mild,Fluent,TCU02a_33.248_7.114.wav,7.114,14
4,40362,47984,ahorita puedo hablar pero flr dos horas flr flr no puedo decir lo todo,male,42.0,TCU02a.wav,77.3,Anomic,Mild,Fluent,TCU02a_40.362_7.622.wav,7.622,14
...,...,...,...,...,...,...,...,...,...,...,...,...,...
647,1922351,1932580,escoge el pan flr y el peanut butters úntelo en el pan,female,53,TCU10a.wav,77.6,,Unknown,NaN,TCU10a_1922.351_10.229.wav,10.229,12
648,1932580,1945033,y después en el otro pan úntelo en el jellys jellys,female,53,TCU10a.wav,77.6,,Unknown,NaN,TCU10a_1932.58_12.453.wav,12.453,11
649,1945033,1949627,y juntelo y ya está,female,53,TCU10a.wav,77.6,,Unknown,NaN,TCU10a_1945.033_4.594.wav,4.594,5
650,1953138,1954175,lau,female,53,TCU10a.wav,77.6,,Unknown,NaN,TCU10a_1953.138_1.037.wav,1.037,1


In [113]:
# Mapeos definidos para convertir texto en números
mapeo_aphasia_type = {
    "Anomic": 5,
    "Conduction": 4,
    "Fluent": 2,
    "Wernicke": 7,
    "TransSensory": 7,
    "Broca": 1,
    "Global": 3,
    "TransMotor": 6,
    "NotAphasicByWAB": 0
}

mapeo_fluency_speech = {
    "Fluent": "Fluente",
    "Non Fluent": "No Fluente",
    "Unknown": -1
}

# Mapeo definido para el género
mapeo_genere = {
    "male": 1,
    "female": 2
}

df_combined['sex_numeric'] = df_combined['sex'].map(mapeo_genere)
df_combined['TipusAfàsia'] = df_combined['aphasia_type'].map(mapeo_aphasia_type)
df_combined['fluency_speech_numeric'] = df_combined['fluency_speech'].map(mapeo_fluency_speech)

df_combined

,mark_start,mark_end,transcriptions,sex,age,file,WAB_AQ,aphasia_type,WAB_AQ_category,fluency_speech,file_cut,duration,num_words,sex_numeric,TipusAfàsia,fluency_speech_numeric
0,20585,24500,okay dímelo otra vez,male,42.0,TCU02a.wav,77.3,Anomic,Mild,Fluent,TCU02a_20.585_3.915.wav,3.915,4,1,5.0,Fluente
1,25710,26400,sí,male,42.0,TCU02a.wav,77.3,Anomic,Mild,Fluent,TCU02a_25.71_0.69.wav,0.690,1,1,5.0,Fluente
2,29400,31200,que piensa mi habla,male,42.0,TCU02a.wav,77.3,Anomic,Mild,Fluent,TCU02a_29.4_1.8.wav,1.800,4,1,5.0,Fluente
3,33248,40362,no es que flr flr decir las cosas a vveces tengo problemas como ahorita,male,42.0,TCU02a.wav,77.3,Anomic,Mild,Fluent,TCU02a_33.248_7.114.wav,7.114,14,1,5.0,Fluente
4,40362,47984,ahorita puedo hablar pero flr dos horas flr flr no puedo decir lo todo,male,42.0,TCU02a.wav,77.3,Anomic,Mild,Fluent,TCU02a_40.362_7.622.wav,7.622,14,1,5.0,Fluente
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
647,1922351,1932580,escoge el pan flr y el peanut butters úntelo en el pan,female,53,TCU10a.wav,77.6,,Unknown,NaN,TCU10a_1922.351_10.229.wav,10.229,12,2,NaN,NaN
648,1932580,1945033,y después en el otro pan úntelo en el jellys jellys,female,53,TCU10a.wav,77.6,,Unknown,NaN,TCU10a_1932.58_12.453.wav,12.453,11,2,NaN,NaN
649,1945033,1949627,y juntelo y ya está,female,53,TCU10a.wav,77.6,,Unknown,NaN,TCU10a_1945.033_4.594.wav,4.594,5,2,NaN,NaN
650,1953138,1954175,lau,female,53,TCU10a.wav,77.6,,Unknown,NaN,TCU10a_1953.138_1.037.wav,1.037,1,2,NaN,NaN


In [114]:
column_mapping = {
    'mark_start': 'Inicio',
    'mark_end': 'Fin',
    'file': 'Transcrip_name',
    'transcriptions': 'Marca',
    'sex_numeric': 'Gènere',
    'age': 'Edat',
    'file_cut': 'name_chunk_audio',
    'WAB_AQ': 'QA',
    'WAB_AQ_category': 'Grup',
    'fluency_speech_numeric': 'Fluente/No Fluente',
    'duration': 'Duración'
}

# Renombrar las columnas del DataFrame df_aphbank
df_combined.rename(columns=column_mapping, inplace=True)


audio_base_path = '/Users/monicaromero/PycharmProjects/afasia_cat/aphasiabank_es/Audios_ES/'
# Crear la columna name_chunk_audio_path concatenando la ruta base con name_chunk_audio
df_combined['name_chunk_audio_path'] = audio_base_path + df_combined['name_chunk_audio']

df_combined['Transcrip_name'] = df_combined['Transcrip_name'].str.replace('.wav', '', regex=False)

df_combined[['Inicio', 'Fin', 'Marca', 'Transcrip_name', 'Duración',
       'name_chunk_audio', 'name_chunk_audio_path', 'Gènere',
       'TipusAfàsia', 'Edat', 'Grup', 'QA', 'Fluente/No Fluente']]


,Inicio,Fin,Marca,Transcrip_name,Duración,name_chunk_audio,name_chunk_audio_path,Gènere,TipusAfàsia,Edat,Grup,QA,Fluente/No Fluente
0,20585,24500,okay dímelo otra vez,TCU02a,3.915,TCU02a_20.585_3.915.wav,/Users/monicaromero/PycharmProjects/afasia_cat/aphasiabank_es/Audios_ES/TCU02a_20.585_3.915.wav,1,5.0,42.0,Mild,77.3,Fluente
1,25710,26400,sí,TCU02a,0.690,TCU02a_25.71_0.69.wav,/Users/monicaromero/PycharmProjects/afasia_cat/aphasiabank_es/Audios_ES/TCU02a_25.71_0.69.wav,1,5.0,42.0,Mild,77.3,Fluente
2,29400,31200,que piensa mi habla,TCU02a,1.800,TCU02a_29.4_1.8.wav,/Users/monicaromero/PycharmProjects/afasia_cat/aphasiabank_es/Audios_ES/TCU02a_29.4_1.8.wav,1,5.0,42.0,Mild,77.3,Fluente
3,33248,40362,no es que flr flr decir las cosas a vveces tengo problemas como ahorita,TCU02a,7.114,TCU02a_33.248_7.114.wav,/Users/monicaromero/PycharmProjects/afasia_cat/aphasiabank_es/Audios_ES/TCU02a_33.248_7.114.wav,1,5.0,42.0,Mild,77.3,Fluente
4,40362,47984,ahorita puedo hablar pero flr dos horas flr flr no puedo decir lo todo,TCU02a,7.622,TCU02a_40.362_7.622.wav,/Users/monicaromero/PycharmProjects/afasia_cat/aphasiabank_es/Audios_ES/TCU02a_40.362_7.622.wav,1,5.0,42.0,Mild,77.3,Fluente
...,...,...,...,...,...,...,...,...,...,...,...,...,...
647,1922351,1932580,escoge el pan flr y el peanut butters úntelo en el pan,TCU10a,10.229,TCU10a_1922.351_10.229.wav,/Users/monicaromero/PycharmProjects/afasia_cat/aphasiabank_es/Audios_ES/TCU10a_1922.351_10.229.wav,2,NaN,53,Unknown,77.6,NaN
648,1932580,1945033,y después en el otro pan úntelo en el jellys jellys,TCU10a,12.453,TCU10a_1932.58_12.453.wav,/Users/monicaromero/PycharmProjects/afasia_cat/aphasiabank_es/Audios_ES/TCU10a_1932.58_12.453.wav,2,NaN,53,Unknown,77.6,NaN
649,1945033,1949627,y juntelo y ya está,TCU10a,4.594,TCU10a_1945.033_4.594.wav,/Users/monicaromero/PycharmProjects/afasia_cat/aphasiabank_es/Audios_ES/TCU10a_1945.033_4.594.wav,2,NaN,53,Unknown,77.6,NaN
650,1953138,1954175,lau,TCU10a,1.037,TCU10a_1953.138_1.037.wav,/Users/monicaromero/PycharmProjects/afasia_cat/aphasiabank_es/Audios_ES/TCU10a_1953.138_1.037.wav,2,NaN,53,Unknown,77.6,NaN


In [115]:
df_combined.to_csv('/Users/monicaromero/PycharmProjects/afasia_cat/aphasiabank_es/df_ES_clean_updated.csv', index=False)